# 05 · Impairment attribution
Rule engine vs LightGBM vs ensemble; SHAP; accuracy vs ground truth.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60)
from networkanalysis.db.database import query_df, table_counts
from networkanalysis.pipeline.features import build_site_feature_table, KPI_DIRECTION, HEADLINE_KPIS

In [ ]:
from networkanalysis.analytics import scoring
from networkanalysis.analytics.attribution import attribute, rule_attribution
from networkanalysis.analytics.groundtruth import load_incidents
feat = build_site_feature_table()
inc = load_incidents()
sc, _ = scoring.compute_scorecard(feat)
attr, report = attribute(feat, inc, sc)

In [ ]:
import json; print(json.dumps(report['final_vs_truth_all'], indent=1))

In [ ]:
# confusion matrix
ev = report['final_vs_truth_all']
import numpy as np
cm = np.array(ev['confusion_matrix'])
plt.imshow(cm, cmap="Blues"); plt.xticks(range(len(ev['labels'])), ev['labels']); plt.yticks(range(len(ev['labels'])), ev['labels'])
for (i,j),v in np.ndenumerate(cm): plt.text(j,i,v,ha="center")
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("final ensemble vs ground truth")

In [ ]:
# rule vs ML agreement
attr.groupby(["rule_class","ml_class"]).size().unstack(fill_value=0)

In [ ]:
# a worked example: top transport site, its evidence
row = attr[attr.final_class=="transport"].merge(sc[["site_id","impact_score"]]).sort_values("impact_score").iloc[-1]
print(row.site_id, "->", row.final_class, round(row.final_confidence,2))
print("\n".join(json.loads(row.rule_evidence)))
print("SHAP:", row.ml_top_features)